In [ ]:
# Setup: CLIP needs open_clip_torch (and internet to download the weights on Kaggle).
# Safe to re-run — pip skips it if already installed.
import importlib.util
if importlib.util.find_spec("open_clip") is None:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
print("open_clip ready")

# Phase 2 — adapting a frozen CLIP to the long tail (Tip-Adapter + LIFT)

The from-scratch leaderboard tops out around `cmo` ~0.47 bal-acc — about the ceiling for
training a small net from scratch on CIFAR-100-LT (IF=100). To go further we stop training a
backbone from scratch and instead **adapt a frozen foundation model (CLIP)** to our long-tailed
data. Two methods, both on **cached** CLIP features (the backbone is run only once):

- **Tip-Adapter** — *training-free* retrieval: classify by similarity to a cache of training
  features, blended with CLIP's zero-shot text prediction. `alpha`/`beta` tuned on val.
- **Tip-Adapter-F** — the same cache, lightly fine-tuned (Balanced Softmax) for a few epochs.
- **LIFT** — a tiny residual adapter (<1% params) + a cosine head initialised from the class-name
  text features, trained with a logit-adjusted loss. Starts at zero-shot CLIP, improves the tail.

These belong in the **external-knowledge (VLM) table**, alongside `vlm_fusion` — not the
from-scratch leaderboard. Needs `!pip install open_clip_torch` and internet on Kaggle.

## 1. Make `src/` importable

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Config

`VAL_FRACTION` / `SEED` must match `run_all_methods.ipynb` so the val split is identical.

In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"
OUTPUT_DIR = PROJECT_DIR / "outputs"

NUM_CLASSES = 100
BATCH_SIZE = 128
NUM_WORKERS = 2
DEVICE = None
SEED = 42
VAL_FRACTION = 0.1

# CLIP backbone. ViT-B-32 is the light default; ViT-B-16 / ViT-L-14 score higher (heavier).
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "openai"
CLIP_PROMPT = "a photo of a {}"

# LIFT hyperparameters (training is on cached features, so these are seconds/epoch).
LIFT_EPOCHS = 50
LIFT_LR = 1e-3
LIFT_BOTTLENECK = 64

# Tip-Adapter-F fine-tuning.
TIPF_EPOCHS = 20
TIPF_LR = 1e-3

# Smoke test: set to e.g. 2000 to cap the cache / training features, None for full.
MAX_TRAIN_SAMPLES = None

# Fine-tuning-depth ablation (HEAVY: backprop through the ViT on images, no feature
# caching). OFF by default; turn on to show that frozen+adapter beats full fine-tuning
# on the tail. Keep FT_EPOCHS small.
RUN_FT_ABLATION = False
FT_EPOCHS = 10

## 3. Imports, device, data splits

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    split_indices_by_class, split_shot_groups, subset)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.experts.clip_expert import (CIFAR100_CLASSES, clip_zero_shot_logits,
                                     encode_clip_features, load_clip)
from src.experts import tip_adapter as ta
from src.experts import lift as lift_mod
from src.utils.experiment import compare_runs, create_run_dir, save_metrics
from src.utils.seed import resolve_device, set_seed

set_seed(SEED)
device = resolve_device(DEVICE)

# class_counts.json is the full LT profile; for the loss we use the actual train-subset counts.
class_counts_full = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts_full)

eval_tf = build_transforms(train=False, image_size=32)  # placeholder; CLIP swaps in its own preprocess
base_train = load_split(DATA_DIR, "train", eval_tf)
train_idx, val_idx = split_indices_by_class([l for _, l in base_train.samples], VAL_FRACTION, SEED)
train_eval = subset(base_train, train_idx)
val_eval = subset(base_train, val_idx)
test_eval = load_split(DATA_DIR, "test", eval_tf)
print("device:", device, "| train", len(train_eval), "| val", len(val_eval), "| test", len(test_eval))


def record(name, result):
    """Compute + save metrics for one method, print the headline numbers, return them."""
    metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
    run_dir = create_run_dir(OUTPUT_DIR, name)
    save_metrics(run_dir, metrics)
    print(f"\n=== {name} ===")
    print(format_metrics(metrics))
    return metrics

## 4. Encode frozen CLIP features once (reused by every method)

The backbone runs exactly once per split. Everything below trains only on these cached
feature tensors, so it is fast enough for a Kaggle session.

In [ ]:
clip_bundle = load_clip(CIFAR100_CLASSES, device, model_name=CLIP_MODEL,
                        pretrained=CLIP_PRETRAINED, prompt=CLIP_PROMPT)

f_train, y_train = encode_clip_features(train_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
f_val, y_val = encode_clip_features(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
f_test, y_test = encode_clip_features(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)

# Smoke test: keep a RANDOM subset (features are ordered by class, so [:N] would keep only
# the first few head classes — random keeps the long-tail shape across all 100 classes).
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(y_train):
    keep = torch.randperm(len(y_train), generator=torch.Generator().manual_seed(SEED))[:MAX_TRAIN_SAMPLES]
    f_train, y_train = f_train[keep], y_train[keep]

# Per-class training counts AFTER the val split (the distribution the loss should correct).
train_counts = list(np.bincount(y_train.numpy(), minlength=NUM_CLASSES))

# Zero-shot CLIP logits for each split (cosine sim to text prototypes, on CLIP's scale).
clip_train = clip_zero_shot_logits(f_train, clip_bundle)
clip_val = clip_zero_shot_logits(f_val, clip_bundle)
clip_test = clip_zero_shot_logits(f_test, clip_bundle)

record("clip_only", {"y_true": y_test.numpy(), "y_pred": clip_test.argmax(1).numpy()})
print("feature dim:", f_train.shape[1])

## 5. Tip-Adapter (training-free) + Tip-Adapter-F (fine-tuned)

Build the cache, pick `alpha`/`beta` on the balanced val accuracy, report on test. Then
fine-tune the cache keys for a few epochs with Balanced Softmax.

In [ ]:
keys, values = ta.build_cache(f_train, y_train, NUM_CLASSES)

alpha, beta, val_acc = ta.tune_alpha_beta(f_val, keys, values, clip_val, y_val.numpy())
print(f"tuned on val -> alpha={alpha:.2f}, beta={beta:.2f}, val bal-acc={val_acc:.4f}")
record("tip_adapter", ta.predict(f_test, y_test, keys, values, clip_test, alpha, beta))

tipf = ta.TipAdapterF(keys, values, beta)
ta.train_tip_adapter_f(tipf, f_train, clip_train, y_train, alpha, train_counts, device,
                       epochs=TIPF_EPOCHS, lr=TIPF_LR)
record("tip_adapter_f",
       ta.predict(f_test, y_test, keys, values, clip_test, alpha, beta, model=tipf, device=device))

## 6. LIFT (adapter + text-initialised cosine head, logit-adjusted loss)

Trains only the tiny adapter and the cosine head; keeps the best epoch by **balanced**
accuracy on val (the project's selection rule).

In [ ]:
lift_model, info = lift_mod.train_lift(
    f_train, y_train, f_val, y_val,
    clip_bundle["text_features"].float(), clip_bundle["logit_scale"],
    train_counts, NUM_CLASSES, device,
    epochs=LIFT_EPOCHS, lr=LIFT_LR, bottleneck=LIFT_BOTTLENECK)
print("best val balanced-acc:", round(info["best_val_balanced_accuracy"], 4))

run_dir = create_run_dir(OUTPUT_DIR, "lift")
pd.DataFrame(info["history"]).to_csv(run_dir / "metrics.csv", index=False)
record("lift", lift_mod.predict(lift_model, f_test, y_test, device))

## 6b. (optional) Fine-tuning-depth ablation — does training CLIP help?

Everything above keeps CLIP **frozen**. Here we sweep how much of the ViT we actually train:
`linear_probe` (head only) -> `last_block` (head + last transformer block) -> `full_ft` (whole
visual encoder), each with Balanced Softmax, best epoch on val. The expected result — tail
accuracy peaks at light adaptation and **drops for full fine-tuning** — is what justifies the
frozen design (LIFT, ICML 2024). HEAVY: set `RUN_FT_ABLATION=True` and expect a few min/epoch.

In [ ]:
if RUN_FT_ABLATION:
    import numpy as np
    from src.experts import clip_finetune
    feat_dim = f_train.shape[1]
    ft_counts = list(np.bincount([l for _, l in train_eval.samples], minlength=NUM_CLASSES))
    lr_by_depth = {"linear_probe": 1e-3, "last_block": 1e-4, "full_ft": 1e-5}
    print("fine-tuning-depth ablation (heavy; trains the ViT on images):")
    for depth in clip_finetune.DEPTHS:
        net, info = clip_finetune.train_clip_finetune(
            clip_bundle["model"], clip_bundle["preprocess"], train_eval, val_eval,
            NUM_CLASSES, ft_counts, feat_dim, device, depth,
            epochs=FT_EPOCHS, lr=lr_by_depth[depth], batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        print(f"  {depth:13s} {info['trainable_params']:>10,} params  val bal={info['best_val_balanced_accuracy']:.4f}")
        record(f"ft_{depth}", clip_finetune.predict(net, test_eval, clip_bundle["preprocess"], device, BATCH_SIZE, NUM_WORKERS))
else:
    print("RUN_FT_ABLATION=False -> skipping the heavy fine-tuning ablation")

## 7. External-knowledge (VLM) comparison table

All CLIP-based methods side by side. Present this as a separate 'external knowledge'
table, not mixed into the from-scratch leaderboard.

In [ ]:
vlm_methods = ["clip_only", "tip_adapter", "tip_adapter_f", "lift", "vlm_fusion",
               "ft_linear_probe", "ft_last_block", "ft_full_ft"]
comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

vlm_table = comparison[comparison["method"].isin(vlm_methods)].reset_index(drop=True)
vlm_table.to_csv(OUTPUT_DIR / "comparison_vlm.csv", index=False)
visualize.plot_metric_comparison(comparison, OUTPUT_DIR)
vlm_table